# 04 - Evaluation and error analysis

This notebook checks ranking quality with a time split. It also looks at user segments, catalogue coverage and the largest ranking errors.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from app.data import load_interactions, load_properties
from app.data.split import temporal_split
from app.evaluation.error_analysis import find_ranking_errors
from app.evaluation.ranking_metrics import (
    catalog_coverage, ndcg_at_k, precision_at_k, recall_at_k,
)
from app.features.user_features import user_preferences_from_history
from app.pipelines.evaluate_model import evaluate
from app.pipelines.train_ranker import build_training_data
from app.recommendation.recommender import PropertyRecommender

properties = load_properties()
interactions = load_interactions()
k = min(10, len(properties))
print(f'Evaluation cutoff: {k}')

## Time-based split

The last interaction from each user is the test item. All earlier interactions stay in training.

In [ ]:
train, test = temporal_split(interactions)
split_summary = pd.Series({
    'train_rows': len(train),
    'test_rows': len(test),
    'train_users': train['user_id'].nunique(),
    'test_users': test['user_id'].nunique(),
    'train_end': train['timestamp'].max(),
    'test_start': test['timestamp'].min(),
})
display(split_summary.to_frame('value'))
display(test.head())

## Compare the weighted and learned rankers

Precision checks how many returned items are relevant. Recall checks how many relevant items were found. MAP and NDCG also reward a good order.

In [ ]:
metrics = evaluate(
    k=k,
    max_users=min(500, interactions['user_id'].nunique()),
    max_training_users=min(1500, interactions['user_id'].nunique()),
)
metrics

In [ ]:
comparison = pd.DataFrame({
    'weighted_baseline': {
        f'Precision@{k}': metrics[f'baseline_precision@{k}'],
        f'Recall@{k}': metrics[f'baseline_recall@{k}'],
        f'MAP@{k}': metrics[f'baseline_map@{k}'],
        f'NDCG@{k}': metrics[f'baseline_ndcg@{k}'],
        'Catalog coverage': metrics['baseline_catalog_coverage'],
    },
    'lightgbm_ranker': {
        f'Precision@{k}': metrics[f'learned_precision@{k}'],
        f'Recall@{k}': metrics[f'learned_recall@{k}'],
        f'MAP@{k}': metrics[f'learned_map@{k}'],
        f'NDCG@{k}': metrics[f'learned_ndcg@{k}'],
        'Catalog coverage': metrics['learned_catalog_coverage'],
    },
})
display(comparison.round(4))
comparison.plot(kind='bar', figsize=(11, 5), ylim=(0, 1), title='Offline ranking metrics')
plt.ylabel('Score')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## User-level results

Average metrics can hide weak results for users with little history. The same metrics are calculated for each user.

In [ ]:
baseline = PropertyRecommender(properties, train)
user_rows = []
recommendation_lists = []

for user_id in test['user_id'].astype(str).unique():
    preferences = user_preferences_from_history(properties, train, user_id)
    # Do not use hard budget or room filters in offline evaluation.
    preferences.update({
        'query': 'Berlin ' + ' '.join(preferences.get('amenities', [])),
        'max_budget': None,
        'bedrooms': None,
        'property_types': [],
    })
    recommended = [item['id'] for item in baseline.recommend(preferences, user_id=user_id, top_k=k)]
    relevant = set(test.loc[test['user_id'].astype(str) == user_id, 'property_id'].astype(str))
    history_size = int((train['user_id'].astype(str) == user_id).sum())
    recommendation_lists.append(recommended)
    user_rows.append({
        'user_id': user_id,
        'history_size': history_size,
        'precision': precision_at_k(recommended, relevant, k),
        'recall': recall_at_k(recommended, relevant, k),
        'ndcg': ndcg_at_k(recommended, relevant, k),
        'recommended': recommended,
        'relevant': sorted(relevant),
    })

user_results = pd.DataFrame(user_rows)
display(user_results.head(10))

In [ ]:
user_results['segment'] = np.select(
    [user_results['history_size'] <= 1, user_results['history_size'] <= 3],
    ['new', 'growing'],
    default='active',
)
segment_results = (
    user_results.groupby('segment', as_index=False)
    .agg(
        users=('user_id', 'count'),
        mean_history=('history_size', 'mean'),
        precision=('precision', 'mean'),
        recall=('recall', 'mean'),
        ndcg=('ndcg', 'mean'),
    )
)
display(segment_results.round(3))

## Coverage and result mix

A model can have fair accuracy while repeating the same popular listings. Coverage checks whether more of the catalogue is used.

In [ ]:
unique_recommended = {item for values in recommendation_lists for item in values}
recommended_properties = properties[properties['property_id'].astype(str).isin(unique_recommended)]
coverage_summary = pd.Series({
    'catalog_coverage': catalog_coverage(recommendation_lists, len(properties)),
    'unique_recommended_listings': len(unique_recommended),
    'recommended_neighborhoods': recommended_properties['neighborhood'].nunique(),
    'catalog_neighborhoods': properties['neighborhood'].nunique(),
    'mean_recommended_price': recommended_properties['price'].mean(),
    'mean_catalog_price': properties['price'].mean(),
})
display(coverage_summary.round(3).to_frame('value'))
display(recommended_properties['neighborhood'].value_counts().head(10).rename('recommended_listings').to_frame())

## Inspect the largest ranking errors

A positive item with a low score is a missed positive. An unobserved item with a high score may be a false positive, but it is not a confirmed dislike.

In [ ]:
error_frame, error_labels, error_groups = build_training_data(
    max_users=min(300, train['user_id'].nunique()),
    candidate_count=min(40, len(properties)),
    min_user_interactions=1,
    properties_frame=properties,
    interactions_frame=train,
)
error_frame = error_frame.copy()
error_frame['label'] = error_labels
error_frame['ranking_score'] = (
    error_frame['semantic_score'] * 0.32
    + error_frame['collaborative_score'] * 0.18
    + error_frame['budget_score'] * 0.18
    + error_frame['amenity_score'] * 0.12
    + error_frame['quality_score'] * 0.12
    + error_frame['popularity_score'] * 0.05
    + error_frame['availability_score'] * 0.03
)
largest_errors = find_ranking_errors(error_frame).head(15)
display(largest_errors[[
    'property_id', 'title', 'label', 'ranking_score', 'error',
    'semantic_score', 'collaborative_score', 'budget_score', 'amenity_score',
]].round(3))

In [ ]:
baseline_ndcg = metrics[f'baseline_ndcg@{k}']
learned_ndcg = metrics[f'learned_ndcg@{k}']
if learned_ndcg > baseline_ndcg:
    print(f'LightGBM wins on NDCG@{k}: {learned_ndcg:.4f} vs {baseline_ndcg:.4f}')
else:
    print(f'Keep the weighted baseline: {baseline_ndcg:.4f} vs {learned_ndcg:.4f}')

## Final decision rules

- Do not promote LightGBM unless it beats the baseline on NDCG and MAP.
- Check new users separately because they have less personal data.
- Track coverage so popular listings do not take every position.
- Add impression, click, save and contact events before making strong business claims.
- Confirm offline gains with an online A/B test before a full release.